# 04 — Post-intervention gambling outcomes

Reproduces Table 5 (significant outcomes), Tables S6-S7 (full 18-cell grids with
rank-biserial effect sizes), and Table S8 (per-arm distributional diagnostics for
significant findings).

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import kruskal, mannwhitneyu, false_discovery_control
import scikit_posthocs as sp

ARM_COL = "CONTROL_GROUP"
ARM_MAP = {"AA": "A", "BB": "B", "CC": "C", "EE": "E", "FF": "F", "CONTROLGROUP": "Control"}
OUTCOMES = {
    "SUM_DEPOSITS_W": "Sum deposits", "NR_DEPOSITS_W": "Number deposits",
    "SUM_WAGERS_DAILY_W": "Sum wagers", "N_WAGERS_DAILY_W": "Number wagers",
    "NR_DAYS_W": "Gambling days", "SUM_TL_W": "Theoretical loss",
}
PERIODS = {"Week 1": [1], "Weeks 2-12": list(range(2, 13)), "Weeks 40-51": list(range(40, 52))}

def psum(df, prefix, weeks):
    cols = [f"{prefix}_POST_{w}" for w in weeks if f"{prefix}_POST_{w}" in df.columns]
    return df[cols].sum(axis=1, min_count=1)

def rbc(ctrl, arm):
    U, _ = mannwhitneyu(ctrl, arm, alternative="two-sided")
    return 1 - (2 * U) / (len(ctrl) * len(arm))

## Full 18-cell grid, pooled effect size, and per-arm medians (Tables S6-S7)

In [ ]:
def full_grid(path, label, arms):
    df = pd.read_csv(path, low_memory=False)
    df["Arm"] = df[ARM_COL].map(ARM_MAP).fillna(df[ARM_COL])
    rows = []
    for pf, on in OUTCOMES.items():
        for pn, wk in PERIODS.items():
            y = psum(df, pf, wk)
            sub = pd.DataFrame({"arm": df["Arm"], "y": y}).dropna()
            g = [sub.loc[sub.arm == a, "y"].values for a in arms + ["Control"]]
            H, p = kruskal(*g)
            ctrl = sub.loc[sub.arm == "Control", "y"].values
            interv = sub.loc[sub.arm.isin(arms), "y"].values
            rb = rbc(ctrl, interv)
            meds = {a: sub.loc[sub.arm == a, "y"].median() for a in arms + ["Control"]}
            row = {"Outcome": on, "Period": pn, "H": round(H, 2), "p_raw": p, "rb_pooled": round(rb, 3)}
            row.update({f"Mdn_{a}": round(meds[a], 1) for a in arms + ["Control"]})
            rows.append(row)
    res = pd.DataFrame(rows)
    res["p_BH"] = false_discovery_control(res["p_raw"].values, method="bh")
    res["sig"] = res["p_BH"] < .05
    print(f"\n===== {label}: FULL 18-CELL GRID =====")
    print(res.assign(p_raw=res.p_raw.round(4), p_BH=res.p_BH.round(4)).to_string(index=False))
    return res

r1 = full_grid("P10_final.csv", "Experiment 1", ["A", "C", "E", "F"])
r2 = full_grid("P11_final.csv", "Experiment 2", ["A", "B", "C"])

## Post-hoc comparisons and per-arm distributional diagnostics for BH-survivors (Table 5, S8)

In [ ]:
def posthoc_and_diagnostics(path, label, arms, res):
    df = pd.read_csv(path, low_memory=False)
    df["Arm"] = df[ARM_COL].map(ARM_MAP).fillna(df[ARM_COL])
    print(f"\n===== {label}: BH-SURVIVORS =====")
    for _, r in res[res.sig].iterrows():
        pf = [k for k, v in OUTCOMES.items() if v == r.Outcome][0]
        wks = PERIODS[r.Period]
        y = psum(df, pf, wks)
        sub = pd.DataFrame({"arm": df["Arm"], "y": y}).dropna()
        dunn = sp.posthoc_dunn(sub, val_col="y", group_col="arm", p_adjust="holm")
        ctrl = sub.loc[sub.arm == "Control", "y"].values
        mc = np.median(ctrl)
        print(f"\n--- {r.Outcome} | {r.Period} (p_BH={r.p_BH:.4f}) ---")
        for a in arms:
            arm = sub.loc[sub.arm == a, "y"].values
            print(f"  Control vs {a}: Dunn_p={dunn.loc['Control', a]:.4f} | "
                  f"median {mc:.0f} vs {np.median(arm):.0f} | "
                  f"mean {ctrl.mean():.1f} vs {arm.mean():.1f} | "
                  f"p99 {np.quantile(ctrl,.99):.0f} vs {np.quantile(arm,.99):.0f} | "
                  f"max {ctrl.max():.0f} vs {arm.max():.0f} | "
                  f"rb={rbc(ctrl, arm):+.3f}")

posthoc_and_diagnostics("P10_final.csv", "Experiment 1", ["A", "C", "E", "F"], r1)
posthoc_and_diagnostics("P11_final.csv", "Experiment 2", ["A", "B", "C"], r2)